In [ ]:
from google.colab import drive, userdata
import json
import os
import sys

drive.mount('/content/drive')

# ----------------------------------------------------------------------
# PARÂMETROS DO PROCESSAMENTO EM LOTE
# ----------------------------------------------------------------------
TABLE_ID = "DIA"
FORCE_UPDATE = True          # True para garantir a versão mais recente do repo
MAX_SESSIONS_PER_RUN = 15    # Recomendado <= 15 para o limite de 20 RPD do Free Tier
SLEEP_SECONDS = 15           # Controle para o limite de 5 RPM do Free Tier
OVERWRITE_EXISTING = False   # False pula sessões que já têm _log.md em archive/logs/<TABLE>/
MODEL_OVERRIDE = None        # None usa a fila padrão de config/gemini_config.json
# ----------------------------------------------------------------------

PROJECT_FOLDER = "/content/drive/MyDrive/rpg-transcription"

if not os.path.exists('rpg_transcription'):
    !git clone https://github.com/ThaisMosken/rpg_transcription.git
elif FORCE_UPDATE:
    !git -C rpg_transcription pull

sys.path.append('/content/rpg_transcription')

from utils.session_helper import build_folder_layout
from src.session_archiver import ensure_folder_with_notice

layout = build_folder_layout(PROJECT_FOLDER, TABLE_ID)
ensure_folder_with_notice(layout["archive_logs_folder"])

from utils.glossary_manager import GlossaryManager
manager = GlossaryManager(TABLE_ID)
context_glossary = manager.get_full_glossary()

with open(os.path.join(layout["config_folder"], "notion_config.json"), "r", encoding="utf-8") as f:
    NOTION_CONFIG = json.load(f)

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NOTION_TOKEN = userdata.get('NOTION_TOKEN')

print(f"✅ Ambiente pronto para o backlog da mesa '{TABLE_ID}'.")

In [ ]:
from src.session_logger import process_backlog_logs

process_backlog_logs(
    api_key=GEMINI_API_KEY,
    chronicles_folder=layout["archive_chronicles_folder"],
    transcriptions_folder=layout["archive_transcriptions_folder"],
    logs_folder=layout["archive_logs_folder"],
    context_glossary=context_glossary,
    table_id=TABLE_ID,
    notion_token=NOTION_TOKEN,
    notion_config=NOTION_CONFIG,
    template="template_log_v1",
    model=MODEL_OVERRIDE,
    batch_size=20,
    sleep_seconds=SLEEP_SECONDS,
    max_sessions_per_run=MAX_SESSIONS_PER_RUN,
    overwrite=OVERWRITE_EXISTING
)